In [0]:
%sql
-- DataSentry v2 setup: an environment fully separate from the live v1 pipeline.
-- Safe to re-run: every statement uses IF NOT EXISTS.

USE CATALOG workspace;

CREATE SCHEMA IF NOT EXISTS dq_bronze COMMENT 'DataSentry v2: raw landing files and Bronze Delta tables';
CREATE SCHEMA IF NOT EXISTS dq_silver COMMENT 'DataSentry v2: validated records with rule flags';
CREATE SCHEMA IF NOT EXISTS dq_gold   COMMENT 'DataSentry v2: scores, exception and remediation logs, clean exports';
CREATE SCHEMA IF NOT EXISTS dq_config COMMENT 'DataSentry v2: dataset configs and rule registry';

CREATE VOLUME IF NOT EXISTS dq_bronze.landing COMMENT 'Raw input files, one folder per dataset. The v2 File Arrival trigger watches this path.';
CREATE VOLUME IF NOT EXISTS dq_config.configs COMMENT 'Dataset YAML configs';
CREATE VOLUME IF NOT EXISTS dq_gold.exports   COMMENT 'Clean dataset exports. Never watched by a trigger.';

SHOW SCHEMAS IN workspace LIKE 'dq_*';

In [0]:
# Copy every banking CSV (batch 1 and batch 2) from the v1 Volume into the v2 landing folder.
# v2 then processes exactly the same input as v1, so results can be compared in Step 4.

SRC = "/Volumes/workspace/banking_datasentry/datasentry_files/"
DST = "/Volumes/workspace/dq_bronze/landing/banking/"

dbutils.fs.mkdirs(DST)

copied = []
for f in dbutils.fs.ls(SRC):
    if f.name.endswith(".csv"):
        dbutils.fs.cp(f.path, DST + f.name)
        copied.append((f.name, f.size))

print(f"{'FILE':<30} {'SIZE (KB)':>10}")
for name, size in copied:
    print(f"{name:<30} {size/1024:>10,.0f}")
print(f"\n{len(copied)} files copied to {DST}")

In [0]:
%pip install "databricks-sdk>=0.89.0" "psycopg[binary]>=3.1.0" -q
dbutils.library.restartPython()

In [0]:
# Lakebase connection test
# Auth: your Databricks identity + a short-lived OAuth token. No password stored anywhere.

import psycopg
from databricks.sdk import WorkspaceClient

LAKEBASE_HOST = "ep-wild-voice-d826qzop.database.us-east-2.cloud.databricks.com"
LAKEBASE_DB   = "databricks_postgres"

w = WorkspaceClient()
user = w.current_user.me().user_name

# Find the endpoint path: projects/<project>/branches/<branch>/endpoints/<endpoint>
endpoints = []
for p in w.postgres.list_projects():
    for b in w.postgres.list_branches(parent=p.name):
        for e in w.postgres.list_endpoints(parent=b.name):
            endpoints.append(e.name)
            print("Found endpoint:", e.name)

LAKEBASE_ENDPOINT = endpoints[0]
token = w.postgres.generate_database_credential(endpoint=LAKEBASE_ENDPOINT).token

with psycopg.connect(host=LAKEBASE_HOST, port=5432, dbname=LAKEBASE_DB,
                     user=user, password=token, sslmode="require",
                     connect_timeout=15) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version(), current_user, now()")
        version, who, ts = cur.fetchone()

print("\nCONNECTED")
print(f"  Postgres : {version.split(',')[0]}")
print(f"  User     : {who}")
print(f"  Time     : {ts}")

In [0]:
# Spark JDBC read from Lakebase: the same pattern Bronze will use for extraction

jdbc_url = f"jdbc:postgresql://{LAKEBASE_HOST}:5432/{LAKEBASE_DB}?sslmode=require"

df = (spark.read.format("jdbc")
      .option("url", jdbc_url)
      .option("user", user)
      .option("password", token)
      .option("query", "SELECT 1 AS ok, current_database() AS db")
      .load())

df.show()